# Empirical analyses

This notebook organizes the Italy, Texas, LA-MRSA, vaccination, and figure analyses. Public inputs are included in `data/`; restricted Meta Colocation Maps are not. Put authorized mobility files in `data/` using the names documented in `data/README.md`. A section with missing inputs reports them and stops cleanly rather than fabricating values.

The next cell loads the reusable numerical implementation from `numerical_method.ipynb`.

In [ ]:
%run numerical_method.ipynb

from pathlib import Path
import pandas as pd

DATA = Path("data")
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)


def require_files(*names):
    paths = [DATA / name for name in names]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        print("Section skipped; missing:", ", ".join(missing))
        return None
    return paths


def read_labeled_matrix(path):
    frame = pd.read_csv(path, index_col=0)
    if frame.shape[0] != frame.shape[1] or list(frame.index.astype(str)) != list(frame.columns.astype(str)):
        raise ValueError(f"{path} must be a labeled square matrix with identical row/column order")
    return frame.index.astype(str).to_list(), validate_C(frame.to_numpy(float))

## Italy

The public province populations and modeled global air-passenger flows are included. The restricted input is `italy_colocation.csv` (Meta, ADM2/NUTS 3, week 13 of 2023). Colocation probabilities are converted using destination population,

$$C_{ij}\propto C^{\mathrm{coloc}}_{ij}n_j,$$

and then normalized to spectral radius one. An optional prepared `italy_importation.csv` has columns `stratum,weight`; the airport-to-province crosswalk needed to rebuild it was not present in the inspected source files.

In [ ]:
italy_inputs = require_files("italy_colocation.csv", "italy_population.csv")
italy_results = None
italy_importation = None
if italy_inputs:
    italy_names, italy_colocation = read_labeled_matrix(italy_inputs[0])
    population = (
        pd.read_csv(italy_inputs[1]).set_index("stratum")
        .loc[italy_names, "population"].to_numpy(float)
    )
    C_italy = normalize_C(italy_colocation * population[np.newaxis, :])
    italy_results = complex_critical_points(C_italy)
    print(f"Recovered {len(italy_results)} retained critical points")

    importation_path = DATA / "italy_importation.csv"
    if importation_path.exists():
        italy_importation = (
            pd.read_csv(importation_path).set_index("stratum")
            .loc[italy_names, "weight"].to_numpy(float)
        )
        italy_importation /= italy_importation.sum()
    else:
        print("Vaccination ranking unavailable: add data/italy_importation.csv")

## Texas

The repository includes the Texas DSHS 2024–2025 county MMR coverage and JHU 2025 county measles burden. The mobility input `texas_colocation.csv` is restricted Meta data (county level, week 9 of 2025). The population vector and optional reference-partition crosswalk follow the interfaces in `data/README.md`.

Susceptibility is applied on the receiving-stratum (row) side, preserving the reproduction-operator convention. Loving County is `NR` in the DSHS source and remains missing; the code raises a clear error if an authorized mobility matrix includes it, rather than silently imputing coverage.

In [ ]:
tx_vaccination = pd.read_csv(DATA / "texas_vaccination.csv")
tx_cases = pd.read_csv(DATA / "texas_measles_cases.csv")
print(f"Public Texas inputs: {len(tx_vaccination)} counties; {tx_cases.cases_2025.sum()} reported cases in 2025")

texas_inputs = require_files("texas_colocation.csv", "texas_population.csv")
texas_results = None
if texas_inputs:
    texas_names, texas_colocation = read_labeled_matrix(texas_inputs[0])
    tx_population = (
        pd.read_csv(texas_inputs[1]).set_index("stratum")
        .loc[texas_names, "population"].to_numpy(float)
    )
    tx_coverage = (
        tx_vaccination.set_index("stratum")
        .loc[texas_names, "coverage"].to_numpy(float)
    )
    if np.any(~np.isfinite(tx_coverage)):
        missing = np.asarray(texas_names)[~np.isfinite(tx_coverage)]
        raise ValueError(f"MMR coverage is unreported for: {', '.join(missing)}")
    C_texas = normalize_C(
        np.diag(1.0 - tx_coverage)
        @ (texas_colocation * tx_population[np.newaxis, :])
    )
    texas_results = complex_critical_points(C_texas)
    print(f"Recovered {len(texas_results)} retained critical points")

## LA-MRSA

The included matrix is Table S1 of Porphyre et al. (2012). Its rows describe contacts made by members of population $i$ with population $j$. The paper's operator instead defines $K_{ij}$ as potentially infectious contacts generated in receiving group $i$ by an individual in source group $j$, so the table is transposed explicitly before computing

$$C=K/\rho(K).$$

Two published cells are censored as `<0.001`. The code uses the reported upper bound `0.001` explicitly and announces this approximation; the source strings remain unchanged in the CSV.

In [ ]:
mrsa_path = DATA / "la_mrsa_contact_matrix.csv"
source_contacts = pd.read_csv(mrsa_path, index_col=0, dtype=str)
if source_contacts.shape[0] != source_contacts.shape[1] or list(source_contacts.index) != list(source_contacts.columns):
    raise ValueError("LA-MRSA source table must have identical row/column group order")

censored = source_contacts.apply(lambda column: column.str.startswith("<")).to_numpy()
contacts_numeric = source_contacts.replace(r"^<0\.001$", "0.001", regex=True).astype(float)
if censored.any():
    print(f"Using the published upper bound 0.001 for {censored.sum()} censored contact values")

mrsa_names = source_contacts.index.to_list()
K = contacts_numeric.to_numpy().T  # receiving group i by infectious source group j
C_mrsa = normalize_C(K)
mrsa_results = complex_critical_points(C_mrsa)
print(f"Recovered {len(mrsa_results)} retained critical points")

## Vaccination analysis

With perfect instantaneous immunity, vaccinating $V_i$ of $n_i$ residents leaves $s_i=1-V_i/n_i$ susceptible and changes the operator by row scaling,

$$R_{ij}^{(\mathbf V)}=s_i(V_i)R_{ij}.$$

The importation-weighted epidemic probability is $p(r,\mathbf V)=\sum_iW_ip_i(r,\mathbf V)$ and effectiveness is $\mathcal E=1-p(r,\mathbf V)/p(r,\mathbf 0)$. For the fragmented-criticality ranking, a province is associated with the critical point on which it has maximal localization and

$$\mathrm{CFI}_i(r)=\frac{W_i}{\sqrt{(r-\operatorname{Re}r_i^c)^2+(\operatorname{Im}r_i^c)^2}}.$$

These definitions preserve the manuscript's row scaling and critical-point distance.

In [ ]:
def vaccinated_C(C, vaccinated, population):
    C = validate_C(C)
    vaccinated = np.asarray(vaccinated, dtype=float)
    population = np.asarray(population, dtype=float)
    if np.any(population <= 0) or np.any(vaccinated < 0) or np.any(vaccinated > population):
        raise ValueError("vaccinated counts must lie between zero and population")
    return np.diag(1.0 - vaccinated / population) @ C


def weighted_epidemic_probability(C, r, importation_weights):
    weights = np.asarray(importation_weights, dtype=float)
    weights = weights / weights.sum()
    return float(weights @ epidemic_probability(C, r))


def campaign_effectiveness(C, r, vaccinated, population, importation_weights):
    baseline = weighted_epidemic_probability(C, r, importation_weights)
    after = weighted_epidemic_probability(vaccinated_C(C, vaccinated, population), r, importation_weights)
    return np.nan if baseline == 0 else 1.0 - after / baseline


def critical_fragility_index(importation_weights, associated_critical_points, r):
    weights = np.asarray(importation_weights, dtype=float)
    points = np.asarray(associated_critical_points, dtype=complex)
    distance = np.sqrt((r - points.real)**2 + points.imag**2)
    return weights / distance


if italy_results is None or italy_importation is None:
    print("Vaccination comparison skipped because the Italy mobility/importation inputs are unavailable.")

## Figure generation

The compact helper below plots retained critical points and mode strength. It writes only generated outputs to `figures/`. Paper panels requiring restricted or other source data are produced only after those inputs have been supplied.

In [ ]:
def plot_critical_landscape(results, title, output_name=None):
    if not results:
        print(f"{title}: no results to plot")
        return
    points = np.array([item["r_c"] for item in results])
    strength = np.array([item["localization"].max() for item in results])
    fig, ax = plt.subplots(figsize=(5.2, 4.0))
    scatter = ax.scatter(points.real, points.imag, c=strength, cmap="viridis", vmin=0, vmax=1)
    ax.set(xlabel=r"$\mathrm{Re}(r_c)$", ylabel=r"$\mathrm{Im}(r_c)$", title=title)
    fig.colorbar(scatter, ax=ax, label="maximum mode localization")
    fig.tight_layout()
    if output_name:
        fig.savefig(FIGURES / output_name, dpi=300, bbox_inches="tight")
    plt.show()


plot_critical_landscape(italy_results, "Italy", "italy_critical_landscape.png")
plot_critical_landscape(texas_results, "Texas", "texas_critical_landscape.png")
plot_critical_landscape(mrsa_results, "LA-MRSA", "la_mrsa_critical_landscape.png")